# AI-Based Student Performance Prediction & Academic Support System
## Stage 1: Dataset Inspection & Exploratory Data Analysis (EDA)

**Dataset Source:** [UCI Machine Learning Repository — Student Performance](https://archive.ics.uci.edu/dataset/320/student+performance)  
**Authors:** Paulo Cortez and Alice Silva (2008), University of Minho, Portugal  
**License:** Creative Commons Attribution 4.0 International (CC BY 4.0)

---
### Objectives of this Inspection:
1. Verify data integrity (rows, columns, data types, missing values, duplicates) for both Mathematics (`student-mat.csv`) and Portuguese (`student-por.csv`).
2. Examine the distribution of the original final grade (`G3`, 0–20 scale).
3. Evaluate the proposed 4-category performance mapping (*Excellent*, *Good*, *Average*, *Needs Improvement*).
4. Analyze class distributions and potential class imbalance.
5. Identify actionable behavioral features vs. demographic factors for non-judgmental recommendations.
6. Formalize data leakage prevention and define two distinct prediction scenarios:
   - **Early-Warning Model:** Inputs collected before any trimester exams (excludes G1, G2, G3).
   - **Mid-Term Model:** Diagnostic model incorporating Period 1 grade G1 (excludes G2, G3).


In [1]:
import pandas as pd
import numpy as np

# Display all columns during inspection
pd.set_option('display.max_columns', 35)
print("Libraries imported successfully.")


Libraries imported successfully.


---
## Part 1: Loading & Inspecting Raw Datasets Separately

The data is loaded directly from the Git-ignored `../data/` directory.
As per project guidelines, the Mathematics and Portuguese datasets are inspected **separately** and are not merged at this initial stage.


In [2]:
# Load datasets (using semicolon separator as specified by UCI)
mat_df = pd.read_csv('../data/student-mat.csv', sep=';')
por_df = pd.read_csv('../data/student-por.csv', sep=';')

print(f"Mathematics Dataset Shape: {mat_df.shape[0]} rows, {mat_df.shape[1]} columns")
print(f"Portuguese Dataset Shape:  {por_df.shape[0]} rows, {por_df.shape[1]} columns")


Mathematics Dataset Shape: 395 rows, 33 columns
Portuguese Dataset Shape:  649 rows, 33 columns


In [3]:
print("--- Mathematics Dataset Integrity ---")
print(f"Missing Values: {mat_df.isnull().sum().sum()}")
print(f"Duplicate Rows: {mat_df.duplicated().sum()}")

print("\n--- Portuguese Dataset Integrity ---")
print(f"Missing Values: {por_df.isnull().sum().sum()}")
print(f"Duplicate Rows: {por_df.duplicated().sum()}")


--- Mathematics Dataset Integrity ---
Missing Values: 0
Duplicate Rows: 0

--- Portuguese Dataset Integrity ---
Missing Values: 0
Duplicate Rows: 0


In [4]:
# Summary of features and data types
dtypes_df = pd.DataFrame({
    'Mathematics_Dtype': mat_df.dtypes,
    'Portuguese_Dtype': por_df.dtypes
})
dtypes_df


Out[0]: 
           Mathematics_Dtype Portuguese_Dtype
school                   str              str
sex                      str              str
age                    int64            int64
address                  str              str
famsize                  str              str
Pstatus                  str              str
Medu                   int64            int64
Fedu                   int64            int64
Mjob                     str              str
Fjob                     str              str
reason                   str              str
guardian                 str              str
traveltime             int64            int64
studytime              int64            int64
failures               int64            int64
schoolsup                str              str
famsup                   str              str
paid                     str              str
activities               str              str
nursery                  str              str
higher                   

Mathematics_Dtype Portuguese_Dtype
school                   str              str
sex                      str              str
age                    int64            int64
address                  str              str
famsize                  str              str
Pstatus                  str              str
Medu                   int64            int64
Fedu                   int64            int64
Mjob                     str              str
Fjob                     str              str
reason                   str              str
guardian                 str              str
traveltime             int64            int64
studytime              int64            int64
failures               int64            int64
schoolsup                str              str
famsup                   str              str
paid                     str              str
activities               str              str
nursery                  str              str
higher                   str              str

---
## Part 2: Distribution of Original Final Grade (`G3`)

The original target variable in both datasets is `G3` (final year evaluation grade), scored on an integer scale from **0 to 20** in the Portuguese education system.


In [5]:
g3_summary = pd.DataFrame({
    'Mathematics (G3)': mat_df['G3'].describe(),
    'Portuguese (G3)': por_df['G3'].describe()
})
print("Summary Statistics for G3:")
print(g3_summary.round(2))

print(f"\nMathematics zero-grade count (G3 == 0): {(mat_df['G3'] == 0).sum()} ({(mat_df['G3'] == 0).mean()*100:.2f}%)")
print(f"Portuguese zero-grade count (G3 == 0):  {(por_df['G3'] == 0).sum()} ({(por_df['G3'] == 0).mean()*100:.2f}%)")


Summary Statistics for G3:
       Mathematics (G3)  Portuguese (G3)
count            395.00           649.00
mean              10.42            11.91
std                4.58             3.23
min                0.00             0.00
25%                8.00            10.00
50%               11.00            12.00
75%               14.00            14.00
max               20.00            19.00

Mathematics zero-grade count (G3 == 0): 38 (9.62%)
Portuguese zero-grade count (G3 == 0):  15 (2.31%)


---
## Part 3: Proposed 4-Category Performance Mapping

### Proposed Mapping Justification:
The target of our system is to classify students into four academic tiers:
- **Excellent:** $G_3 \in [16, 20]$ (Corresponds to Class A in European/Portuguese grading)
- **Good:** $G_3 \in [14, 15]$ (Corresponds to Class B)
- **Average:** $G_3 \in [10, 13]$ (Corresponds to Satisfactory/Sufficient passing grades C & D)
- **Needs Improvement:** $G_3 \in [0, 9]$ (Below the official Portuguese pass threshold of 10; Class F)

*Note: This mapping is a documented project design decision aligning with our approved SRS, rather than an official UCI-prescribed taxonomy.*


In [6]:
def map_performance_category(g3):
    if g3 >= 16:
        return 'Excellent'
    elif g3 >= 14:
        return 'Good'
    elif g3 >= 10:
        return 'Average'
    else:
        return 'Needs Improvement'

# Apply categorization
mat_df['performance_category'] = mat_df['G3'].apply(map_performance_category)
por_df['performance_category'] = por_df['G3'].apply(map_performance_category)

categories = ['Excellent', 'Good', 'Average', 'Needs Improvement']

mat_counts = mat_df['performance_category'].value_counts()[categories]
mat_pcts = (mat_df['performance_category'].value_counts(normalize=True)[categories] * 100).round(2)

por_counts = por_df['performance_category'].value_counts()[categories]
por_pcts = (por_df['performance_category'].value_counts(normalize=True)[categories] * 100).round(2)

category_dist = pd.DataFrame({
    'Math Count': mat_counts,
    'Math %': mat_pcts,
    'Portuguese Count': por_counts,
    'Portuguese %': por_pcts
})
category_dist


Out[0]: 
                      Math Count  Math %  Portuguese Count  Portuguese %
performance_category                                                    
Excellent                     40   10.13                82         12.63
Good                          60   15.19               112         17.26
Average                      165   41.77               355         54.70
Needs Improvement            130   32.91               100         15.41


Math Count  Math %  Portuguese Count  Portuguese %
performance_category                                                    
Excellent                     40   10.13                82         12.63
Good                          60   15.19               112         17.26
Average                      165   41.77               355         54.70
Needs Improvement            130   32.91               100         15.41


---
## Part 4: Class Imbalance & Feature Relationship Analysis

From the distribution above:
- In **Mathematics**, `Average` is 41.77% and `Needs Improvement` is 32.91%, while `Excellent` is only 10.13%.
- In **Portuguese**, `Average` represents a majority (54.70%), with `Needs Improvement` at 15.41% and `Excellent` at 12.63%.
- **Imbalance Consideration:** Models evaluated on these categories must use balanced metrics (such as Macro F1-score or Weighted F1-score) rather than raw accuracy alone.


In [7]:
print("=== Cross-Tabulation: Past Failures vs Performance (Mathematics, in %) ===")
failures_ct = (pd.crosstab(mat_df['failures'], mat_df['performance_category'], normalize='index') * 100).round(2)[categories]
print(failures_ct)

print("\n=== Cross-Tabulation: Weekly Study Time vs Performance (Mathematics, in %) ===")
# studytime: 1 (<2h), 2 (2-5h), 3 (5-10h), 4 (>10h)
studytime_ct = (pd.crosstab(mat_df['studytime'], mat_df['performance_category'], normalize='index') * 100).round(2)[categories]
print(studytime_ct)

print("\n=== Cross-Tabulation: School Extra Educational Support vs Performance (Mathematics, in %) ===")
schoolsup_ct = (pd.crosstab(mat_df['schoolsup'], mat_df['performance_category'], normalize='index') * 100).round(2)[categories]
print(schoolsup_ct)


=== Cross-Tabulation: Past Failures vs Performance (Mathematics, in %) ===
performance_category  Excellent   Good  Average  Needs Improvement
failures                                                          
0                          12.5  18.59    43.91              25.00
1                           2.0   2.00    44.00              52.00
2                           0.0   5.88    11.76              82.35
3                           0.0   0.00    25.00              75.00

=== Cross-Tabulation: Weekly Study Time vs Performance (Mathematics, in %) ===
performance_category  Excellent   Good  Average  Needs Improvement
studytime                                                         
1                         10.48  13.33    40.95              35.24
2                          7.58  12.12    44.95              35.35
3                         13.85  26.15    35.38              24.62
4                         18.52  18.52    37.04              25.93

=== Cross-Tabulation: School Extra Educa

---
## Part 5: Data Leakage Prevention & Two Prediction Scenarios

### Correlation Analysis
Let's examine the correlation of intermediate grades (`G1`, `G2`) versus non-grade features with `G3`.


In [8]:
num_cols_mat = mat_df.select_dtypes(include=[np.number]).columns
mat_corr = mat_df[num_cols_mat].corr()['G3'].sort_values(ascending=False)

print("Correlation with Final Grade (G3) in Mathematics:")
print(mat_corr[['G2', 'G1', 'Medu', 'Fedu', 'studytime', 'failures', 'absences', 'Dalc', 'Walc']].round(4))


Correlation with Final Grade (G3) in Mathematics:
G2           0.9049
G1           0.8015
Medu         0.2171
Fedu         0.1525
studytime    0.0978
failures    -0.3604
absences     0.0342
Dalc        -0.0547
Walc        -0.0519
Name: G3, dtype: float64


### Analysis of Leakage & Two Prediction Scenarios:
1. **The Leakage Danger:** `G2` has a correlation of **+0.905** (Math) and **+0.919** (Portuguese) with `G3`. Including `G2` gives models an almost deterministic shortcut, rendering behavioral features useless.
2. **Scenario 1 — Early-Warning Model (Primary Stage 2/3 Goal):**
   - **Excluded features:** `G1`, `G2`, and `G3`.
   - **Target:** `performance_category` (derived from `G3`).
   - **Practical utility:** Provides early guidance at enrollment / start of the semester before any formal testing occurs.
3. **Scenario 2 — Mid-Term Diagnostic Model:**
   - **Included features:** `G1` (Period 1 test) + behavioral features.
   - **Excluded features:** `G2` and `G3`.
   - **Target:** `performance_category` (derived from `G3`).
   - **Practical utility:** Provides revised academic intervention after initial assessment tests.


---
## Part 6: Actionable Features for Supportive Recommendations

### What the Data Supports:
The dataset provides clear, non-judgmental variables for rule-based academic support:
1. `studytime`: Recommend moving from Level 1/2 (<5 hrs/week) to Level 3/4 structured study plans.
2. `absences`: Trigger early notifications when absence count exceeds threshold levels (>10 sessions).
3. `failures`: Flag students with prior academic difficulty for institutional tutoring.
4. `schoolsup` & `famsup`: Guide students towards existing remedial academic tutoring programs.

### Explicit Limitations (Boundaries of the Dataset):
- **No Topic-Level Granularity:** The dataset does not distinguish topics (e.g., algebra vs. geometry, or grammar vs. composition). We must not claim subtopic weak-area detection.
- **Correlation $\ne$ Causation:** Variables like `schoolsup` correlate negatively with grades simply because struggling students are targeted for school support. Recommendations must be supportive rather than punitive.
- **Ethical Safeguards:** Demographic and socio-economic variables (`sex`, `address`, `Mjob`, `Pstatus`) must not be used to justify adverse or deterministic judgments about student potential.


---
## Part 7: Note on Cross-Subject Analysis & Linking

The UCI dataset contains an official matching script (`student-merge.R`) identifying 382 students who took both courses, matched on 11 non-grade attributes:
`['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason']`.

**Limitation of Merging:**
- Matching without unique student IDs can yield minor ambiguities or false matches if multiple students share identical demographic profiles.
- For initial modeling, treating Mathematics and Portuguese as separate subject models maintains clean evaluation without assumptions.


In [9]:
print("Stage 1 Dataset Inspection & EDA Notebook successfully verified and executed.")
print("Summary: Zero missing values, zero duplicates, clean 4-tier target defined, leakage boundaries established.")


Stage 1 Dataset Inspection & EDA Notebook successfully verified and executed.
Summary: Zero missing values, zero duplicates, clean 4-tier target defined, leakage boundaries established.
